## Ingest drivers.json


In [0]:
%run "../utils/config"

### Step 1 - Read the json file


#### 1.1 Define the schema


In [0]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, DateType


In [0]:
name_schema = StructType([
    StructField("forename", StringType(), False),
    StructField("surname", StringType(), False)
])


In [0]:
drivers_schema = StructType([
    StructField("code", StringType(), False),
    StructField("dob", DateType(), True),
    StructField("driverId", IntegerType(), True),
    StructField("driverRef", StringType(), True),
    StructField("name", name_schema, True),
    StructField("nationality", StringType(), True),
    StructField("number", IntegerType(), True),
    StructField("url", StringType(), True)
])


#### 1.2 Read the json file


In [0]:
raw_path = raw_folder_path


In [0]:
drivers_df = (
    spark.read
        .format("json")
        .schema(drivers_schema)
        .load(f"{raw_path}/drivers.json")
        )


### Step 2 - Transform the data


In [0]:
from pyspark.sql.functions import col, concat, lit, current_timestamp


In [0]:

drivers_transformed_df = add_ingestion_date(
    drivers_df
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("driverRef", "driver_ref")
        .withColumn("name",concat(col("name.forename"),lit(" "), col("name.surname")))
        )


In [0]:
dirvers_renamed_df = drivers_transformed_df.drop(col("url"))


### Step 3 - Write the output to parquet


In [0]:
processed_path = processed_folder_path


In [0]:
dirvers_renamed_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(f"{processed_path}/drivers")
